# PubMed-RAG-TR: data and model provenance

1. load the released datasets directly from their immutable Hugging Face revisions;
2. validate and summarize the data used by the manuscript;
3. make the 600-example evaluation split reproducible; and
4. connect the released embedding/reranker checkpoints to transparent training inputs and hyperparameters.

Public data: [Pubmed-RAG-TR](https://huggingface.co/datasets/SMARTICT/Pubmed-RAG-TR) and [Pubmed-RAG-TR-LLM-Eval](https://huggingface.co/datasets/SMARTICT/Pubmed-RAG-TR-LLM-Eval). No local CSV or Google Drive path is required.

## Execution guide

The dataset audit and split construction run on CPU in a few minutes. Model training is intentionally disabled by default because it requires a CUDA GPU and several hours. Set `RUN_TRAINING = True` only after reviewing the provenance notes below.

The final translated/QA dataset is treated as the released research artifact. Re-running translation and synthetic QA generation would not be deterministic and would incur external API cost; the prompts are already printed in Appendix A of the manuscript. This notebook therefore begins from the released Hugging Face snapshots.

In [ ]:
%pip install -q "datasets==3.6.0" "pandas==2.2.3" "numpy==1.26.4" "scikit-learn==1.6.1" "sentence-transformers==4.1.0" "transformers==4.51.3" "accelerate==1.6.0"


In [ ]:
from __future__ import annotations

import ast
import hashlib
import json
import os
import random
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict, load_dataset
from IPython.display import display

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

@dataclass(frozen=True)
class StudyConfig:
    dataset_id: str = "SMARTICT/Pubmed-RAG-TR"
    dataset_revision: str = "7c6016f"
    judge_dataset_id: str = "SMARTICT/Pubmed-RAG-TR-LLM-Eval"
    judge_dataset_revision: str = "cc66365"
    legacy_test_fraction: float = 0.06
    manuscript_eval_size: int = 600
    seed: int = SEED
    embedding_base: str = "NeuML/pubmedbert-base-embeddings"
    embedding_release: str = "SMARTICT/pubmedbert-base-embeddings-tr-pubmed-10k"
    reranker_base_v1: str = "Alibaba-NLP/gte-multilingual-reranker-base"
    reranker_release_v1: str = "SMARTICT/gte-multilingual-reranker-base-pubmed-tr-v1"
    reranker_base_v2: str = "Alibaba-NLP/gte-reranker-modernbert-base"
    reranker_release_v2: str = "SMARTICT/gte-reranker-modernbert-base-pubmed-tr-v1"

CFG = StudyConfig()
RUN_TRAINING = False
RUN_MODEL_SMOKE_TEST = False
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

pd.Series(asdict(CFG), name="value").to_frame()

## 1. Load immutable public snapshots

In [ ]:
hf_token = os.getenv("HF_TOKEN") or None

source_ds = load_dataset(
    CFG.dataset_id,
    revision=CFG.dataset_revision,
    split="train",
    token=hf_token,
)
judge_ds = load_dataset(
    CFG.judge_dataset_id,
    revision=CFG.judge_dataset_revision,
    split="train",
    token=hf_token,
)

print(source_ds)
print(judge_ds)
print("source fingerprint:", source_ds._fingerprint)
print("judge fingerprint: ", judge_ds._fingerprint)

In [ ]:
SOURCE_COLUMNS = {"id", "Questions", "Answer", "tr_contents", "PMID"}
JUDGE_COLUMNS = {"id", "PMID", "en_contents", "tr_contents", "evaluation", "total_rating", "error"}

assert SOURCE_COLUMNS.issubset(source_ds.column_names), source_ds.column_names
assert JUDGE_COLUMNS.issubset(judge_ds.column_names), judge_ds.column_names
assert len(source_ds) == 10_001, "The pinned source revision is expected to contain 10,001 rows."
assert source_ds.unique("id") and len(source_ds.unique("id")) == len(source_ds)

pd.DataFrame(
    {
        "artifact": [CFG.dataset_id, CFG.judge_dataset_id],
        "revision": [CFG.dataset_revision, CFG.judge_dataset_revision],
        "rows": [len(source_ds), len(judge_ds)],
        "columns": [", ".join(source_ds.column_names), ", ".join(judge_ds.column_names)],
    }
)

## 2. Normalize QA fields

In [ ]:
def parse_list(value: Any) -> list[str]:
    if isinstance(value, np.ndarray):
        value = value.tolist()
    if isinstance(value, (list, tuple)):
        parsed = list(value)
    elif value is None or (isinstance(value, float) and np.isnan(value)):
        parsed = []
    elif isinstance(value, str):
        text = value.strip()
        if not text:
            parsed = []
        else:
            try:
                parsed = json.loads(text)
            except json.JSONDecodeError:
                parsed = ast.literal_eval(text)
    else:
        raise TypeError(f"Unsupported list representation: {type(value)!r}")
    if not isinstance(parsed, list):
        raise ValueError(f"Expected a list, received {type(parsed)!r}")
    return [str(item).strip() for item in parsed if str(item).strip()]

source_df = source_ds.to_pandas()
source_df["questions"] = source_df["Questions"].map(parse_list)
source_df["answers"] = source_df["Answer"].map(parse_list)
source_df["qa_count"] = source_df["questions"].map(len)
source_df["answer_count"] = source_df["answers"].map(len)
source_df["qa_lengths_match"] = source_df["qa_count"] == source_df["answer_count"]

assert source_df["qa_lengths_match"].all()
assert source_df["id"].is_unique
assert source_df["tr_contents"].fillna("").str.strip().ne("").all()

source_df[["id", "PMID", "qa_count", "questions", "answers", "tr_contents"]].head(3)

In [ ]:
dataset_audit = pd.Series(
    {
        "documents": len(source_df),
        "unique_ids": source_df["id"].nunique(),
        "unique_pmids": source_df["PMID"].nunique(),
        "total_qa_pairs": int(source_df["qa_count"].sum()),
        "documents_without_qa": int(source_df["qa_count"].eq(0).sum()),
        "documents_with_mismatched_qa": int((~source_df["qa_lengths_match"]).sum()),
        "median_document_characters": int(source_df["tr_contents"].str.len().median()),
    },
    name="value",
)

display(dataset_audit.to_frame())
display(
    source_df["qa_count"]
    .value_counts(dropna=False)
    .sort_index()
    .rename_axis("QA pairs per document")
    .to_frame("documents")
)

## 3. Reproduce the LLM-as-a-judge summary

In [ ]:
judge_df = judge_ds.to_pandas()
ratings = pd.to_numeric(judge_df["total_rating"], errors="coerce")
labels = ratings.map({1.0: "1", 2.0: "2", 3.0: "3"}).fillna("Not Processed")
quality_table = (
    labels.value_counts()
    .reindex(["1", "2", "3", "Not Processed"], fill_value=0)
    .rename("Sample Count")
    .to_frame()
)
quality_table["Percentage (%)"] = 100 * quality_table["Sample Count"] / len(judge_df)
quality_table.loc["High quality (2 or 3)"] = [
    int(labels.isin(["2", "3"]).sum()),
    100 * labels.isin(["2", "3"]).mean(),
]
quality_table.round({"Percentage (%)": 2})

## 4. Deterministic evaluation split

In [ ]:
legacy_split: DatasetDict = source_ds.train_test_split(
    test_size=CFG.legacy_test_fraction,
    seed=CFG.seed,
)
legacy_test = legacy_split["test"]
paper_test = legacy_test.select(range(CFG.manuscript_eval_size))

assert len(legacy_test) == 601
assert len(paper_test) == 600
assert set(legacy_split["train"]["id"]).isdisjoint(set(paper_test["id"]))

def id_fingerprint(dataset: Dataset) -> str:
    payload = "\n".join(dataset["id"]).encode("utf-8")
    return hashlib.sha256(payload).hexdigest()

split_manifest = {
    "dataset_id": CFG.dataset_id,
    "dataset_revision": CFG.dataset_revision,
    "seed": CFG.seed,
    "legacy_test_fraction": CFG.legacy_test_fraction,
    "legacy_test_rows": len(legacy_test),
    "manuscript_test_rows": len(paper_test),
    "ordered_id_sha256": id_fingerprint(paper_test),
    "excluded_legacy_row_id": legacy_test[-1]["id"],
}
print(json.dumps(split_manifest, indent=2, ensure_ascii=False))
(OUTPUT_DIR / "evaluation_split_manifest.json").write_text(
    json.dumps(split_manifest, indent=2, ensure_ascii=False), encoding="utf-8"
)

In [ ]:
def first_qa_frame(dataset: Dataset) -> pd.DataFrame:
    rows = []
    for record in dataset:
        questions = parse_list(record["Questions"])
        answers = parse_list(record["Answer"])
        if not questions or len(questions) != len(answers):
            continue
        rows.append(
            {
                "id": record["id"],
                "pmid": record["PMID"],
                "context": record["tr_contents"],
                "question": questions[0],
                "reference_answer": answers[0],
            }
        )
    return pd.DataFrame(rows)

def explode_all_qa(frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for row in frame.itertuples(index=False):
        for pair_index, (question, answer) in enumerate(zip(row.questions, row.answers)):
            rows.append(
                {
                    "id": row.id,
                    "pmid": row.PMID,
                    "pair_index": pair_index,
                    "question": question,
                    "answer": answer,
                    "context": row.tr_contents,
                }
            )
    return pd.DataFrame(rows)

evaluation_df = first_qa_frame(paper_test)
all_qa_df = explode_all_qa(source_df)
assert len(evaluation_df) == 600
assert evaluation_df["question"].str.len().gt(0).all()
assert evaluation_df["reference_answer"].str.len().gt(0).all()

print(f"Evaluation rows: {len(evaluation_df):,}")
print(f"All released QA pairs: {len(all_qa_df):,}")
evaluation_df.head(3)

## 5. Training inputs and released checkpoints

The model repositories are the most precise public record of the checkpoints that were actually released. They report:

| Component | Released checkpoint | Base model | Loss | Released-card settings |
|---|---|---|---|---|
| Dense embedding | `SMARTICT/pubmedbert-base-embeddings-tr-pubmed-10k` | `NeuML/pubmedbert-base-embeddings` | Multiple-Negatives Ranking Loss | 5 epochs; batch 32; gradient accumulation 16; LR 2e-5 |
| GTE reranker | `SMARTICT/gte-multilingual-reranker-base-pubmed-tr-v1` | `Alibaba-NLP/gte-multilingual-reranker-base` | BCE (`pos_weight=5`) | 2 epochs; batch 16; LR 2e-5 |
| ModernBERT reranker | `SMARTICT/gte-reranker-modernbert-base-pubmed-tr-v1` | `Alibaba-NLP/gte-reranker-modernbert-base` | BCE (`pos_weight=5`) | 2 epochs; batch 16; LR 2e-5 |


In [ ]:
# One positive (question, source document) pair per document, matching the
# released embedding model card's 10,001-sample training dataset.
embedding_pairs_df = pd.DataFrame(
    {
        "anchor": source_df["questions"].str[0],
        "positive": source_df["tr_contents"],
        "id": source_df["id"],
    }
).dropna(subset=["anchor", "positive"])

assert len(embedding_pairs_df) == 10_001
embedding_pairs = Dataset.from_pandas(embedding_pairs_df[["anchor", "positive"]], preserve_index=False)
embedding_splits = embedding_pairs.train_test_split(test_size=0.10, seed=CFG.seed)
embedding_splits

In [ ]:
if RUN_TRAINING:
    import torch
    from sentence_transformers import (
        SentenceTransformer,
        SentenceTransformerTrainer,
        SentenceTransformerTrainingArguments,
        losses,
    )
    from sentence_transformers.training_args import BatchSamplers

    embedding_model = SentenceTransformer(CFG.embedding_base)
    embedding_loss = losses.MultipleNegativesRankingLoss(embedding_model)
    use_bf16 = bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported())

    embedding_args = SentenceTransformerTrainingArguments(
        output_dir=str(OUTPUT_DIR / "embedding_model"),
        num_train_epochs=5,
        per_device_train_batch_size=32,
        per_device_eval_batch_size=16,
        gradient_accumulation_steps=16,
        learning_rate=2e-5,
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        batch_sampler=BatchSamplers.NO_DUPLICATES,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        bf16=use_bf16,
        tf32=torch.cuda.is_available(),
        seed=CFG.seed,
        report_to="none",
    )
    embedding_trainer = SentenceTransformerTrainer(
        model=embedding_model,
        args=embedding_args,
        train_dataset=embedding_splits["train"],
        eval_dataset=embedding_splits["test"],
        loss=embedding_loss,
    )
    embedding_trainer.train()
    embedding_model.save_pretrained(OUTPUT_DIR / "embedding_model" / "final")
else:
    print("Training skipped. Set RUN_TRAINING=True to run the released-card embedding recipe.")

### Deterministic hard-negative mining

In [ ]:
def mine_hard_negatives(
    pairs: pd.DataFrame,
    encoder: Any,
    negatives_per_query: int = 2,
    query_batch_size: int = 64,
) -> pd.DataFrame:
    required = {"id", "anchor", "positive"}
    if not required.issubset(pairs.columns):
        raise ValueError(f"Expected columns {sorted(required)}")

    doc_embeddings = encoder.encode(
        pairs["positive"].tolist(),
        batch_size=32,
        normalize_embeddings=True,
        show_progress_bar=True,
        convert_to_numpy=True,
    )
    rows: list[dict[str, Any]] = []
    for start in range(0, len(pairs), query_batch_size):
        stop = min(start + query_batch_size, len(pairs))
        queries = pairs["anchor"].iloc[start:stop].tolist()
        query_embeddings = encoder.encode(
            queries, normalize_embeddings=True, convert_to_numpy=True
        )
        similarities = query_embeddings @ doc_embeddings.T
        for local_index, query in enumerate(queries):
            source_index = start + local_index
            rows.append(
                {"query": query, "document": pairs.iloc[source_index]["positive"], "label": 1.0,
                 "query_id": pairs.iloc[source_index]["id"], "document_id": pairs.iloc[source_index]["id"]}
            )
            order = np.argsort(similarities[local_index])[::-1]
            selected = [idx for idx in order if idx != source_index][:negatives_per_query]
            for document_index in selected:
                rows.append(
                    {"query": query, "document": pairs.iloc[document_index]["positive"], "label": 0.0,
                     "query_id": pairs.iloc[source_index]["id"], "document_id": pairs.iloc[document_index]["id"]}
                )
    return pd.DataFrame(rows)

if RUN_TRAINING:
    from sentence_transformers import SentenceTransformer

    mining_encoder = SentenceTransformer(CFG.embedding_release)
    reranker_rows_df = mine_hard_negatives(embedding_pairs_df, mining_encoder)
    reranker_rows_df.to_json(
        OUTPUT_DIR / "mined_reranker_rows.jsonl", orient="records", lines=True, force_ascii=False
    )
    display(reranker_rows_df["label"].value_counts().sort_index())
else:
    print("Hard-negative mining skipped.")

In [ ]:
if RUN_TRAINING:
    import torch
    from sentence_transformers.cross_encoder import (
        CrossEncoder,
        CrossEncoderTrainer,
        CrossEncoderTrainingArguments,
    )
    from sentence_transformers.cross_encoder.losses import BinaryCrossEntropyLoss

    reranker_dataset = Dataset.from_pandas(
        reranker_rows_df[["query", "document", "label"]], preserve_index=False
    ).train_test_split(test_size=0.10, seed=CFG.seed)

    for base_model, output_name, trust_remote_code in [
        (CFG.reranker_base_v1, "gte_multilingual_reranker", True),
        (CFG.reranker_base_v2, "gte_modernbert_reranker", True),
    ]:
        model = CrossEncoder(base_model, num_labels=1, trust_remote_code=trust_remote_code)
        loss = BinaryCrossEntropyLoss(model=model, pos_weight=torch.tensor(5.0))
        args = CrossEncoderTrainingArguments(
            output_dir=str(OUTPUT_DIR / output_name),
            num_train_epochs=2,
            per_device_train_batch_size=16,
            per_device_eval_batch_size=16,
            learning_rate=2e-5,
            warmup_ratio=0.1,
            eval_strategy="steps",
            save_strategy="steps",
            load_best_model_at_end=True,
            bf16=bool(torch.cuda.is_available() and torch.cuda.is_bf16_supported()),
            seed=CFG.seed,
            report_to="none",
        )
        trainer = CrossEncoderTrainer(
            model=model, args=args, train_dataset=reranker_dataset["train"],
            eval_dataset=reranker_dataset["test"], loss=loss
        )
        trainer.train()
        model.save_pretrained(OUTPUT_DIR / output_name / "final")
else:
    print("Reranker training skipped. Released checkpoints are used in Notebook 02.")

## 6. Released-checkpoint smoke test

In [ ]:
if RUN_MODEL_SMOKE_TEST:
    from sentence_transformers import CrossEncoder, SentenceTransformer

    example = evaluation_df.iloc[0]
    embedding_model = SentenceTransformer(CFG.embedding_release, token=hf_token)
    reranker_v1 = CrossEncoder(CFG.reranker_release_v1, trust_remote_code=True, token=hf_token)
    reranker_v2 = CrossEncoder(CFG.reranker_release_v2, trust_remote_code=True, token=hf_token)

    vectors = embedding_model.encode(
        [example.question, example.context], normalize_embeddings=True, convert_to_numpy=True
    )
    pair = [(example.question, example.context)]
    print("embedding cosine:", float(vectors[0] @ vectors[1]))
    print("reranker V1 score:", float(np.asarray(reranker_v1.predict(pair)).reshape(-1)[0]))
    print("reranker V2 score:", float(np.asarray(reranker_v2.predict(pair)).reshape(-1)[0]))
else:
    print("Checkpoint download skipped. Set RUN_MODEL_SMOKE_TEST=True to enable it.")

## 7. Reference result reported in the manuscript


In [ ]:
embedding_reported = pd.DataFrame(
    {
        "Model": [CFG.embedding_base, CFG.embedding_release],
        "NDCG@10": [0.2653, 0.8713],
        "provenance": ["manuscript Table: embedding result", "released model card / manuscript"],
    }
).set_index("Model")
embedding_reported